In [1]:
from sentence_transformers import SentenceTransformer, util
from sklearn.metrics.pairwise import cosine_similarity
from pprint import pprint
import pandas as pd
import numpy as np

/usr/local/lib/python3.10/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [5]:
model=SentenceTransformer("all-mpnet-base-v2")

In [6]:
data=pd.read_pickle("reviews_cleaned.pickle")

In [7]:
data.shape

(86001, 4)

In [8]:
data=data.head(5000)

In [9]:
data.shape

(5000, 4)

In [10]:
travel=pd.read_excel("updated_complete_dataset.xlsx")

In [11]:
travel.head()

,Place_id,Name,Category,Description,Country,City,Latitude,Longitude,Rating,Number_of_Reviews,Opening_Hours,Entry_Fee,Best_Time_to_Visit,Suggested_Duration,Accessibility,Contact_Info,Popularity_Index,Tags
0,1,Hagia Sophia,Historic Landmark,"A former Greek Orthodox Christian basilica, la...",Turkey,Istanbul,41.0082,28.9784,4.4,3141,9:00 AM - 7:00 PM,28,Spring,2-3 hours,Wheelchair Accessible,contact@example.com,88,"historic, cultural"
1,2,Blue Mosque,Mosque,"A historic mosque in Istanbul, known for its b...",Turkey,Istanbul,41.0085,28.9769,4.7,3081,9:00 AM - 7:00 PM,32,Spring,2-3 hours,Wheelchair Accessible,contact@example.com,90,"historic, cultural"
2,3,Topkapi Palace,Palace,"A large palace in Istanbul, home to Ottoman su...",Turkey,Istanbul,41.0120,28.9794,4.9,4362,9:00 AM - 7:00 PM,9,Spring,2-3 hours,Wheelchair Accessible,contact@example.com,89,"historic, cultural"
3,4,Cappadocia,Natural Wonder,"A unique region in central Turkey, known for i...",Turkey,Nevşehir,38.6737,34.8235,4.2,1466,9:00 AM - 7:00 PM,27,Spring,2-3 hours,Wheelchair Accessible,contact@example.com,71,"historic, cultural"
4,5,Pamukkale,Natural Wonder,Famous for its white travertine terraces creat...,Turkey,Denizli,37.9268,29.1228,4.5,2076,9:00 AM - 7:00 PM,26,Spring,2-3 hours,Wheelchair Accessible,contact@example.com,81,"historic, cultural"


In [12]:
travel.shape

(2666, 18)

In [13]:
data['index']=np.random.randint(1,2667,size=len(data))

In [14]:
data.head()

,hotel,city,content,clean_review,index
0,The Reserve at Paradisus Punta Cana,Punta Cana,Hands down great family vacation. Estefanía ou...,hands down great family vacation estefan our f...,584
1,Paradisus Punta Cana Resort,Punta Cana,Juan Batistae has got to be the best host/bart...,juan batistae has got be the best host bartend...,415
2,Dreams Palm Beach Punta Cana,Punta Cana,This was our first trip! We went with our frie...,this was our first trip went with our friends ...,1966
3,Now Onyx Punta Cana,Punta Cana,Me and my partner travelled from Glasgow Scotl...,me and partner travelled from glasgow scotland...,608
4,Paradisus Punta Cana Resort,Punta Cana,We had amazing time with family and friends. A...,we had amazing time with family and friends an...,2289


In [15]:
data['review_embedding']=data['clean_review'].apply(lambda x:model.encode(x).tolist())

In [90]:
travel['description_embedding'] = travel['Description'].apply(lambda x: model.encode(x if pd.notna(x) else ""))

In [91]:
travel['category_embedding'] = travel['Category'].apply(lambda x: model.encode(x if pd.notna(x) else ""))

In [19]:
np.save("touristic_place_review_embeddings.npy", np.vstack(data["review_embedding"].values))

In [20]:
data.drop(columns=['review_embedding']).to_csv("touristic_place_review_without_embedding.csv",index=False)

In [21]:
np.save("touristic_places_description_embeddings.npy", np.vstack(travel['description_embedding'].values))

In [22]:
np.save("touristic_places_category_embeddings.npy", np.vstack(travel['category_embedding'].values))

In [23]:
model.save_pretrained("touristic_place_review_model")

In [24]:
import pickle
with open('model_and_review_data.pkl','wb') as f:
  pickle.dump((model, data), f)

In [25]:
import pickle
with open('model_review_data_and_travel_data.pkl','wb') as f:
  pickle.dump((model, data, travel), f)

In [124]:
# Travel_id bazında review embedding ortalaması
reviews_grouped = data.groupby('index')['review_embedding'].apply(lambda x: np.mean(np.vstack(x), axis=0))

# Travel datasetine review embedding'leri ekleme
travel = travel.merge(reviews_grouped, left_on='Place_id', right_index=True, how='left')

# Yeni sütunu düzenle
travel.rename(columns={'review_embedding': 'average_review_embedding'}, inplace=True)

# Boş embedding'leri sıfır vektörüyle doldur
travel['average_review_embedding'] = travel['average_review_embedding'].apply(
    lambda x: np.zeros(768) if isinstance(x, float) else x
)


In [125]:
travel.head()

,Place_id,Name,Category,Description,Country,City,Latitude,Longitude,Rating,Number_of_Reviews,...,Entry_Fee,Best_Time_to_Visit,Suggested_Duration,Accessibility,Contact_Info,Popularity_Index,Tags,description_embedding,category_embedding,average_review_embedding
0,1,Hagia Sophia,Historic Landmark,"A former Greek Orthodox Christian basilica, la...",Turkey,Istanbul,41.0082,28.9784,4.4,3141,...,28,Spring,2-3 hours,Wheelchair Accessible,contact@example.com,88,"historic, cultural","[-0.00020653808, -0.036556713, -0.011851492, -...","[-0.0090959845, 0.06781446, -0.005679004, 0.01...","[-0.020912155508995056, -0.015316734206862748,..."
1,2,Blue Mosque,Mosque,"A historic mosque in Istanbul, known for its b...",Turkey,Istanbul,41.0085,28.9769,4.7,3081,...,32,Spring,2-3 hours,Wheelchair Accessible,contact@example.com,90,"historic, cultural","[0.010132032, -0.05600847, -0.015305804, -0.02...","[0.00784914, 0.085784, -0.012818557, -0.011515...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
2,3,Topkapi Palace,Palace,"A large palace in Istanbul, home to Ottoman su...",Turkey,Istanbul,41.0120,28.9794,4.9,4362,...,9,Spring,2-3 hours,Wheelchair Accessible,contact@example.com,89,"historic, cultural","[0.0029206616, -0.01955726, 0.001745966, 0.019...","[0.007207393, 0.11394712, -0.010757201, 0.0299...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
3,4,Cappadocia,Natural Wonder,"A unique region in central Turkey, known for i...",Turkey,Nevşehir,38.6737,34.8235,4.2,1466,...,27,Spring,2-3 hours,Wheelchair Accessible,contact@example.com,71,"historic, cultural","[0.008714523, -0.09297628, -0.015092146, -0.02...","[-0.006392043, 0.0928174, -0.023952091, -0.002...","[-0.05077267996966839, 0.009368178434669971, -..."
4,5,Pamukkale,Natural Wonder,Famous for its white travertine terraces creat...,Turkey,Denizli,37.9268,29.1228,4.5,2076,...,26,Spring,2-3 hours,Wheelchair Accessible,contact@example.com,81,"historic, cultural","[-0.03831799, -0.011627738, -0.032850366, 0.02...","[-0.006392043, 0.0928174, -0.023952091, -0.002...","[-0.03534308262169361, 0.0004090673173777759, ..."


In [236]:
user_input="visited due a friends wedding the entire experience was wonderful staff super friendly clean facilities and beautiful gardens not mention the beach another level live paradise costa rica yet the beaches the caribbean island they are another levelspecial thanks maria the front desk and yuleidi rosa the restaurants great service"

In [237]:
user_embedding=model.encode(user_input)

In [44]:
from sklearn.decomposition import PCA

# Embedding boyutlarını 768'e çıkartmak için PCA kullanmak
pca = PCA(n_components=768)

# Örnek: category_embedding'i 768 boyutuna çıkarmak
category_embedding_768 = pca.fit_transform(np.vstack(travel['category_embedding'].values))

# Benzer işlemi diğer embedding'ler için de uygulayabilirsin


In [46]:
description_embedding_768 = pca.fit_transform(np.vstack(travel['description_embedding'].values))

In [71]:
reviews_grouped.head()

,review_embedding
index,
1,"[-0.020912155508995056, -0.015316734206862748,..."
4,"[-0.05077267996966839, 0.009368178434669971, -..."
5,"[-0.03534308262169361, 0.0004090673173777759, ..."
6,"[-0.031448330730199814, -0.04072212055325508, ..."
7,"[-0.012089982628822327, -0.014756759628653526,..."


In [72]:
average_review_embedding_768 = pca.fit_transform(np.vstack(reviews_grouped.values))

In [247]:
from sklearn.metrics.pairwise import cosine_similarity

category_weight=0.1
description_weight=0.1
review_weight=0.9


travel['similarity_score'] = (
    category_weight * cosine_similarity(np.vstack(travel['category_embedding'].values), [user_embedding])[:, 0] +
    description_weight * cosine_similarity(np.vstack(travel['description_embedding'].values), [user_embedding])[:, 0] +
    review_weight * cosine_similarity(np.vstack(travel['average_review_embedding'].values), [user_embedding])[:, 0]
)

top_places= travel.sort_values(by='similarity_score',ascending=False).head(1)
print(top_places[['Country','similarity_score']])


   Country  similarity_score
20  Turkey            0.8739


In [249]:
import json

# Ağırlıklar
weights = {
    "tags_weight": 0.1,
    "description_weight": 0.1,
    "review_weight": 0.9
}

# Ağırlıkları kaydet
with open("weights.json", "w") as f:
    json.dump(weights, f)

# Transformer modelini kaydet
model.save_pretrained("weighted_and_embedded_cosine_similarity_model")


In [84]:
travel.drop('description_embedding', axis=1, inplace=True)
travel.drop('category_embedding', axis=1, inplace=True)
travel.drop('average_review_embedding', axis=1, inplace=True)

In [87]:
data.head()

,hotel,city,content,clean_review,index,review_embedding
0,The Reserve at Paradisus Punta Cana,Punta Cana,Hands down great family vacation. Estefanía ou...,hands down great family vacation estefan our f...,584,"[-0.0620141364634037, -0.001158999395556748, -..."
1,Paradisus Punta Cana Resort,Punta Cana,Juan Batistae has got to be the best host/bart...,juan batistae has got be the best host bartend...,415,"[-0.04903048276901245, 0.021676143631339073, 0..."
2,Dreams Palm Beach Punta Cana,Punta Cana,This was our first trip! We went with our frie...,this was our first trip went with our friends ...,1966,"[-0.02740250900387764, -0.020269978791475296, ..."
3,Now Onyx Punta Cana,Punta Cana,Me and my partner travelled from Glasgow Scotl...,me and partner travelled from glasgow scotland...,608,"[0.004050245508551598, 0.00817936472594738, -0..."
4,Paradisus Punta Cana Resort,Punta Cana,We had amazing time with family and friends. A...,we had amazing time with family and friends an...,2289,"[-0.04765491932630539, 0.02227485366165638, -0..."


In [88]:
print(type(data['review_embedding'].iloc[0]))

<class 'list'>


In [89]:
embedding_dim = len(data['review_embedding'].iloc[0])
print(f"Embedding boyutu: {embedding_dim}")

Embedding boyutu: 768


In [118]:
travel.head()

,Place_id,Name,Category,Description,Country,City,Latitude,Longitude,Rating,Number_of_Reviews,...,Entry_Fee,Best_Time_to_Visit,Suggested_Duration,Accessibility,Contact_Info,Popularity_Index,Tags,description_embedding,category_embedding,average_review_embedding
0,1,Hagia Sophia,Historic Landmark,"A former Greek Orthodox Christian basilica, la...",Turkey,Istanbul,41.0082,28.9784,4.4,3141,...,28,Spring,2-3 hours,Wheelchair Accessible,contact@example.com,88,"historic, cultural","[-0.00020653808, -0.036556713, -0.011851492, -...","[-0.0090959845, 0.06781446, -0.005679004, 0.01...","[-0.020912155508995056, -0.015316734206862748,..."
1,2,Blue Mosque,Mosque,"A historic mosque in Istanbul, known for its b...",Turkey,Istanbul,41.0085,28.9769,4.7,3081,...,32,Spring,2-3 hours,Wheelchair Accessible,contact@example.com,90,"historic, cultural","[0.010132032, -0.05600847, -0.015305804, -0.02...","[0.00784914, 0.085784, -0.012818557, -0.011515...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
2,3,Topkapi Palace,Palace,"A large palace in Istanbul, home to Ottoman su...",Turkey,Istanbul,41.0120,28.9794,4.9,4362,...,9,Spring,2-3 hours,Wheelchair Accessible,contact@example.com,89,"historic, cultural","[0.0029206616, -0.01955726, 0.001745966, 0.019...","[0.007207393, 0.11394712, -0.010757201, 0.0299...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
3,4,Cappadocia,Natural Wonder,"A unique region in central Turkey, known for i...",Turkey,Nevşehir,38.6737,34.8235,4.2,1466,...,27,Spring,2-3 hours,Wheelchair Accessible,contact@example.com,71,"historic, cultural","[0.008714523, -0.09297628, -0.015092146, -0.02...","[-0.006392043, 0.0928174, -0.023952091, -0.002...","[-0.05077267996966839, 0.009368178434669971, -..."
4,5,Pamukkale,Natural Wonder,Famous for its white travertine terraces creat...,Turkey,Denizli,37.9268,29.1228,4.5,2076,...,26,Spring,2-3 hours,Wheelchair Accessible,contact@example.com,81,"historic, cultural","[-0.03831799, -0.011627738, -0.032850366, 0.02...","[-0.006392043, 0.0928174, -0.023952091, -0.002...","[-0.03534308262169361, 0.0004090673173777759, ..."


In [93]:
embedding_dim = len(travel['category_embedding'].iloc[0])
print(f"Embedding boyutu: {embedding_dim}")

Embedding boyutu: 768


In [94]:
embedding_dim = len(travel['description_embedding'].iloc[0])
print(f"Embedding boyutu: {embedding_dim}")

Embedding boyutu: 768


In [98]:
embedding_dim = len(travel['average_review_embedding'].iloc[0])
print(f"Embedding boyutu: {embedding_dim}")

Embedding boyutu: 768


In [99]:
embedding_dim = len(user_embedding)
print(f"Embedding boyutu: {embedding_dim}")

Embedding boyutu: 768


In [126]:
# Tüm embedding boyutlarını kontrol edin
travel['embedding_size'] = travel['average_review_embedding'].apply(lambda x: len(x) if isinstance(x, (list, np.ndarray)) else None)

# Farklı boyutları listeleyin
print(travel['embedding_size'].value_counts())


embedding_size
768    2666
Name: count, dtype: int64


**Sentiment Analysis**

In [ ]:
from transformers import pipeline, LongformerTokenizer, LongformerForSequenceClassification

# Longformer modelini ve tokenizer'ı yükleyin
model_name = "allenai/longformer-base-4096"
sentiment_pipeline = pipeline("sentiment-analysis", model=model_name, tokenizer=model_name)


data['sentiment'] = data['clean_review'].apply(lambda x: sentiment_pipeline(x)[0]['label'])



config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/597M [00:00<?, ?B/s]

Some weights of LongformerForSequenceClassification were not initialized from the model checkpoint at allenai/longformer-base-4096 and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Initializing global attention on CLS token...
Input ids are automatically padded to be a multiple of `config.attention_window`: 512


model.safetensors:   0%|          | 0.00/597M [00:00<?, ?B/s]